In [1]:
import matplotlib.pyplot as plt
import numpy as np

import archimedes as arc
from archimedes.experimental.approximation import FunctionSpace

## Block-push problem


$$
\min_{u(t), \mathbf{x}} \int_0^1 u^2(\tau) ~ d \tau, \qquad \text{subject to} \qquad
\begin{cases}
\dot{\mathbf{x}} &= f(t, \mathbf{x}, u) \\ 
\mathbf{x}(0) &= \begin{bmatrix} 0 & 0 \end{bmatrix}^T \\
\mathbf{x}(1) &= \begin{bmatrix} 1 & 0 \end{bmatrix}^T \\
\end{cases}
$$

where the dynamics are the double-integrator

$$
f(t, \mathbf{x}, u) = \begin{bmatrix} 0 & 1 \\ 0 & 0 \end{bmatrix} \mathbf{x} + \begin{bmatrix} 0 \\ 1 \end{bmatrix} u
$$

In [5]:
t0, tf = 0.0, 1.0
x0, xf = np.array([0.0, 0.0]), np.array([1.0, 0.0])

n = 10  # number of intervals
breakpoints = np.linspace(t0, tf, n + 1, endpoint=True)

# Set up the Simpson quadrature rule for the collocation method
quad_rule = arc.quadrature.simpson(n, a=t0, b=tf)

# Hermite-Simpson collocation - discretize `u` with piecewise
# linear functions and `x` with piecewise cubic functions
V_u = FunctionSpace.piecewise(
    kind="lagrange",
    degree=1,
    breakpoints=breakpoints,
    nodes="lobatto",
    quad_rule=quad_rule,
)
V_x = FunctionSpace.piecewise(
    kind="hermite",
    degree=3,
    breakpoints=breakpoints,
    quad_rule=quad_rule,
)


@arc.struct
class BlockPushParameters:
    u: np.ndarray  # control coefficients (n,)
    x: np.ndarray  # state coefficients (2, n)



def obj(params: BlockPushParameters) -> float:
    """Objective function for the block-pushing problem."""
    u = V_u.function(params.u)
    return u.dot(u)


u0 = V_u.function()
x0 = V_x.function()
v0 = V_x.function()
init_params = BlockPushParameters(
    u=u0.coefficients,
    x=np.stack([x0.coefficients, v0.coefficients], axis=0),
)

obj(init_params)  # evaluate the objective at the initial guess

np.float64(0.0)

In [3]:
V_u.quad_rule

QuadratureRule(name='gauss_legendre', measure=LegendreMeasure, n=20)